# 01 · Data overview

First look at the three raw Transfermarkt files, restricted to our scope:
**current Premier League players, last 5 seasons.** Nothing is modelled here; we only load, inspect and check data quality.

In [ ]:
import sys
from pathlib import Path

# make `src` importable when the notebook runs from notebooks/
sys.path.append(str(Path.cwd().parent))

import pandas as pd
import matplotlib.pyplot as plt

from src import config
from src.load_data import load_players, load_valuations, load_appearances

pd.set_option('display.max_columns', 40)
print('League:', config.LEAGUE_ID, '| seasons kept from', config.START_DATE)

## 1. Load (already filtered)

In [ ]:
players = load_players()
ids = players['player_id']
valuations = load_valuations(ids)
appearances = load_appearances(ids)

pd.DataFrame(
    {'rows': [len(players), len(valuations), len(appearances)],
     'players': [players['player_id'].nunique(), valuations['player_id'].nunique(), appearances['player_id'].nunique()]},
    index=['players', 'valuations', 'appearances'],
)

`players` is the starting list. Players with no valuations or appearances in the window (e.g. youngsters who never played) will be dealt with in the cleaning notebook.

## 2. Players table

In [ ]:
players.head()

In [ ]:
players.dtypes

In [ ]:
# share of missing values per column, worst first
players.isna().mean().sort_values(ascending=False).round(2).head(12)

In [ ]:
players['current_club_name'].value_counts()

In [ ]:
players['position'].value_counts()

## 3. Market value (our target)

In [ ]:
players['market_value_in_eur'].describe()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))
players['market_value_in_eur'].dropna().plot.hist(bins=40, ax=axes[0], title='Market value (EUR)')
import numpy as np
np.log1p(players['market_value_in_eur'].dropna()).plot.hist(bins=40, ax=axes[1], title='log(1 + market value)')
plt.tight_layout()

The raw value is strongly right-skewed (a few stars worth far more than everyone else); the log version is much closer to a bell shape. We will work with the log value.

## 4. Valuation history

In [ ]:
print(valuations['date'].min().date(), '->', valuations['date'].max().date())
valuations.groupby('player_id').size().describe()

In [ ]:
valuations.groupby(valuations['date'].dt.year).size().plot.bar(title='Valuations recorded per year')

## 5. Appearances

In [ ]:
print(appearances['date'].min().date(), '->', appearances['date'].max().date())
appearances['competition_id'].value_counts().head(10)

In [ ]:
appearances[['goals', 'assists', 'minutes_played', 'yellow_cards', 'red_cards']].describe()